# Gestura — sign recognition trainingTrains the Sign→Speech recogniser on the 40-word ISL vocabulary, then exports acheckpoint the repo loads directly.**Runtime → Change runtime type → T4 GPU** before running (it works on CPU, just slower).### What this trains on`features.npz` holds every clip already run through `pose_features`, the samefunction the runtime classifier uses — 641 clips, 40 glosses, each a`(32, 134)` array: 32 resampled frames of 67 landmarks (25 upper body + 21 + 21hands) in x and y, shoulder-centred and shoulder-width-scaled.Shipping features rather than video is deliberate. It means this notebook needsonly numpy and torch — no `pose-format`, no MediaPipe (which pins`mediapipe<0.10.30` and Python 3.12) — and 8MB instead of 190MB. It also removesthe failure mode where the notebook reimplements feature extraction slightlydifferently, scores well here, and then performs badly at runtime with nothingto show why.### What it is beating| | 40-class top-1 ||---|---|| DTW template matching (what the repo shipped) | 62.1% || AI4Bharat INCLUDE-263, fine-tuned | 47.8% || small GRU on these features | ~70% |INCLUDE's pretrained checkpoints are real and load correctly, but they take**absolute pixel coordinates**, so the model learns where in the frame thesigner stands. This corpus mixes four sources at different resolutions, and itdoes not survive that. These features are framing-invariant by construction.

In [ ]:
import torch, numpy as np, urllib.request, time, json, mathfrom pathlib import PathDEVICE = "cuda" if torch.cuda.is_available() else "cpu"print("torch", torch.__version__, "| device:", DEVICE)if DEVICE == "cuda":    print(torch.cuda.get_device_name(0))else:    print("No GPU — Runtime > Change runtime type > T4 GPU. CPU works, ~4x slower.")SRC = "https://raw.githubusercontent.com/Ajay-1011-git/Gestura/main/data/models/features.npz"if not Path("features.npz").exists():    print("downloading features...")    urllib.request.urlretrieve(SRC, "features.npz")d = np.load("features.npz", allow_pickle=True)X, y, classes = d["X"], d["y"], [str(c) for c in d["classes"]]print(f"\n{X.shape[0]} clips, {len(classes)} classes, each {X.shape[1]}x{X.shape[2]}")counts = np.bincount(y, minlength=len(classes))print("clips per class:", ", ".join(f"{c}({n})" for c, n in zip(classes, counts)))

## SplitTwo clips held out per class where the class can spare them, one where it cannot,and none for the three classes with fewer than three clips — those can betrained on but not honestly tested.Validation for early stopping comes out of the **training** half. Stopping on thetest set reports a number that will not reproduce on anything else, and that isthe single easiest way to fool yourself here.

In [ ]:
def make_split(y, seed=7, val_fraction=0.12):    rng = np.random.RandomState(seed)    test_idx = []    for c in range(len(classes)):        idx = np.where(y == c)[0]        n_test = 2 if len(idx) >= 5 else (1 if len(idx) >= 3 else 0)        test_idx += list(rng.permutation(idx)[:n_test])    test_idx = np.array(sorted(test_idx))    rest = np.array([i for i in range(len(y)) if i not in set(test_idx)])    rest = rng.permutation(rest)    cut = max(1, int(val_fraction * len(rest)))    return rest[cut:], rest[:cut], test_idxtr, va, te = make_split(y)untestable = [classes[c] for c in range(len(classes))              if not np.isin(te, np.where(y == c)[0]).any()]print(f"train {len(tr)}  val {len(va)}  test {len(te)}")print("no test clip (too few):", ", ".join(untestable) or "none")

## Models`SignNet` is the architecture the repo defines in `backend/recognition/neural.py`— it must stay identical or the exported checkpoint will not load. The others arehere to see whether anything does better, which is what the GPU is for.

In [ ]:
import torch.nn as nnclass SignNet(nn.Module):    """Must match backend/recognition/neural.py exactly — the repo loads this."""    def __init__(self, n_classes, hidden=128, layers=2):        super().__init__()        self.gru = nn.GRU(134, hidden, num_layers=layers, batch_first=True,                          bidirectional=True, dropout=0.3)        self.head = nn.Sequential(nn.LayerNorm(hidden*2), nn.Dropout(0.4),                                  nn.Linear(hidden*2, n_classes))    def forward(self, x):        out, _ = self.gru(x)        return self.head(out.max(dim=1).values)   # the distinctive moment, not the last frameclass WideGRU(SignNet):    def __init__(self, n): super().__init__(n, hidden=256, layers=2)class DeepGRU(SignNet):    def __init__(self, n): super().__init__(n, hidden=192, layers=3)class SignTransformer(nn.Module):    def __init__(self, n_classes, d=192, heads=4, layers=3):        super().__init__()        self.proj = nn.Linear(134, d)        self.pos = nn.Parameter(torch.randn(1, 32, d) * 0.02)        enc = nn.TransformerEncoderLayer(d, heads, d*4, dropout=0.3,                                         batch_first=True, norm_first=True)        self.enc = nn.TransformerEncoder(enc, layers)        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.4), nn.Linear(d, n_classes))    def forward(self, x):        h = self.enc(self.proj(x) + self.pos)        return self.head(h.max(dim=1).values)class Conv1DNet(nn.Module):    def __init__(self, n_classes, ch=256):        super().__init__()        self.net = nn.Sequential(            nn.Conv1d(134, ch, 5, padding=2), nn.BatchNorm1d(ch), nn.GELU(), nn.Dropout(0.3),            nn.Conv1d(ch, ch, 3, padding=1), nn.BatchNorm1d(ch), nn.GELU(), nn.Dropout(0.3),            nn.Conv1d(ch, ch, 3, padding=1), nn.BatchNorm1d(ch), nn.GELU(),        )        self.head = nn.Sequential(nn.LayerNorm(ch), nn.Dropout(0.4), nn.Linear(ch, n_classes))    def forward(self, x):        return self.head(self.net(x.transpose(1, 2)).max(dim=2).values)for name, cls in [("SignNet (repo)", SignNet), ("WideGRU", WideGRU),                  ("DeepGRU", DeepGRU), ("SignTransformer", SignTransformer),                  ("Conv1DNet", Conv1DNet)]:    print(f"{name:<18} {sum(p.numel() for p in cls(len(classes)).parameters())/1e3:>7.0f}k params")

## TrainingAugmentation matters more than architecture at 575 clips. The two that help area global scale jitter and a per-clip offset — the same sign filmed closer or alittle off-centre is the same sign, and this corpus varies in both because it isaggregated from four different collections.Time masking is the third: dropping a short span of frames stops the modelkeying on one instant.

In [ ]:
def augment(xb):    xb = xb * (1 + torch.randn(len(xb), 1, 1, device=xb.device) * 0.05)        # scale    xb = xb + torch.randn(len(xb), 1, 134, device=xb.device) * 0.02           # offset    if torch.rand(1).item() < 0.5:                                            # time mask        t0 = torch.randint(0, 26, (1,)).item()        xb = xb.clone(); xb[:, t0:t0+6, :] = 0    return xbdef train_model(model_cls, seed=0, epochs=150, lr=2e-3, quiet=True):    torch.manual_seed(seed); np.random.seed(seed)    Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)    yt = torch.tensor(y, dtype=torch.long).to(DEVICE)    m = model_cls(len(classes)).to(DEVICE)    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-2)    sch = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=epochs*max(1, len(tr)//32))    lossf = nn.CrossEntropyLoss(label_smoothing=0.1)    best_val, best_state = -1.0, None    for ep in range(epochs):        m.train(); perm = np.random.permutation(tr)        for i in range(0, len(perm) - 31, 32):            b = perm[i:i+32]            opt.zero_grad()            lossf(m(augment(Xt[b])), yt[b]).backward()            nn.utils.clip_grad_norm_(m.parameters(), 1.0)            opt.step(); sch.step()        m.eval()        with torch.no_grad():            v = (m(Xt[va]).argmax(1) == yt[va]).float().mean().item()        if v > best_val:            best_val, best_state = v, {k: t.clone() for k, t in m.state_dict().items()}        if not quiet and ep % 30 == 0:            print(f"    epoch {ep:>3}  val {v:.1%}")    m.load_state_dict(best_state); m.eval()    with torch.no_grad():        logits = m(Xt[te])        top1 = (logits.argmax(1) == yt[te]).float().mean().item()        top3 = (logits.topk(3, 1).indices == yt[te][:, None]).any(1).float().mean().item()    return m, best_val, top1, top3, logits.cpu()print("sanity run (SignNet, 1 seed)...")t0 = time.time()_, v, t1, t3, _ = train_model(SignNet, quiet=False)print(f"\nval {v:.1%}  test top-1 {t1:.1%}  top-3 {t3:.1%}   [{time.time()-t0:.0f}s]")

## Which architectureThree seeds each, so the differences reported are bigger than the noise. At 66test clips one clip is 1.5%, so anything under about 4% apart is a tie.

In [ ]:
results = {}for name, cls in [("SignNet (repo)", SignNet), ("WideGRU", WideGRU), ("DeepGRU", DeepGRU),                  ("SignTransformer", SignTransformer), ("Conv1DNet", Conv1DNet)]:    runs = [train_model(cls, seed=s) for s in range(3)]    t1 = np.array([r[2] for r in runs]); t3 = np.array([r[3] for r in runs])    results[name] = dict(top1=t1.mean(), std=t1.std(), top3=t3.mean(),                         logits=[r[4] for r in runs], cls=cls)    print(f"{name:<18} top-1 {t1.mean():.1%} +/- {t1.std():.1%}   top-3 {t3.mean():.1%}")print("\nDTW baseline on this split: 62.1%")

## EnsembleAveraging the logits of the architectures that disagree is usually worth a fewpoints, and costs nothing at inference here — but only take it if it actuallywins, and only export it if the repo can load it.

In [ ]:
yte_t = torch.tensor(y[te], dtype=torch.long)best_single = max(results, key=lambda k: results[k]["top1"])all_logits = [lg for r in results.values() for lg in r["logits"]]ens = torch.stack(all_logits).softmax(-1).mean(0)ens_acc = (ens.argmax(1) == yte_t).float().mean().item()print(f"best single : {best_single} at {results[best_single]['top1']:.1%}")print(f"ensemble    : {ens_acc:.1%}  ({len(all_logits)} models)")print(f"\nDTW baseline: 62.1%  ->  {'+' if ens_acc>0.621 else ''}{(ens_acc-0.621)*100:.1f} points")

## Confidence calibrationThe escalation waterfall keys off confidence, not just the label, so thethreshold has to mean something. `0.22` was calibrated for DTW's *distance*margin; a softmax margin is a different scale and needs its own number.Pick the threshold that holds precision around 85% — above it the systemtranslates, below it it asks. Letting a wrong sign through confidently is muchworse here than asking a question.

In [ ]:
m_final, _, t1, t3, logits = train_model(results[best_single]["cls"], seed=0)probs = logits.softmax(-1)top2 = probs.topk(2, 1).valuesmargin = (top2[:, 0] - top2[:, 1]).numpy()correct = (probs.argmax(1) == yte_t).numpy()print("threshold   kept   precision")chosen = 0.0for t in [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:    keep = correct[margin >= t]    if len(keep) >= 5:        p = keep.mean()        print(f"   {t:.2f}     {len(keep):>3}      {p:.0%}")        if p >= 0.85 and chosen == 0.0:            chosen = tprint(f"\nsuggested LEXICON_HIT threshold: {chosen:.2f}")print("set this in backend/recognition/classifier.py if it differs from 0.22")

## ExportSaved in the shape `NeuralSignClassifier.load()` expects. Only the repo's`SignNet` can be exported — if another architecture won, port it into`backend/recognition/neural.py` first, or the checkpoint will not load.

In [ ]:
if results[best_single]["cls"] is not SignNet:    print(f"NOTE: {best_single} beat SignNet. Exporting SignNet anyway so the")    print("checkpoint loads; port the winner into backend/recognition/neural.py to use it.\n")export_model, val, t1, t3, _ = train_model(SignNet, seed=0)torch.save({"classes": classes, "state": {k: v.cpu() for k, v in export_model.state_dict().items()}},           "recognizer.pt")print(f"exported SignNet: {len(classes)} classes, val {val:.1%}, test top-1 {t1:.1%}, top-3 {t3:.1%}")try:    from google.colab import files    files.download("recognizer.pt")except Exception:    print("not in Colab — download recognizer.pt from the file browser")

## Back in the repo```bash.venv/bin/python scripts/import_recognizer.py ~/Downloads/recognizer.pt```That checks the class list matches the vocabulary on disk, loads the weights,and scores the model on held-out clips before installing it — a checkpoint thatloads is not the same as a checkpoint that works.